In [3]:
import pyiqa
import torch
import os

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [2]:
print(pyiqa.list_models())

['ahiq', 'arniqa', 'arniqa-clive', 'arniqa-csiq', 'arniqa-flive', 'arniqa-kadid', 'arniqa-live', 'arniqa-spaq', 'arniqa-tid', 'brisque', 'brisque_matlab', 'ckdn', 'clipiqa', 'clipiqa+', 'clipiqa+_rn50_512', 'clipiqa+_vitL14_512', 'clipscore', 'cnniqa', 'compare2score', 'cw_ssim', 'dbcnn', 'deepdc', 'dists', 'entropy', 'fid', 'fid_dinov2', 'fsim', 'gmsd', 'hyperiqa', 'ilniqe', 'inception_score', 'laion_aes', 'liqe', 'liqe_mix', 'lpips', 'lpips+', 'lpips-vgg', 'lpips-vgg+', 'mad', 'maniqa', 'maniqa-kadid', 'maniqa-pipal', 'ms_ssim', 'msswd', 'musiq', 'musiq-ava', 'musiq-paq2piq', 'musiq-spaq', 'nima', 'nima-koniq', 'nima-spaq', 'nima-vgg16-ava', 'niqe', 'niqe_matlab', 'nlpd', 'nrqm', 'paq2piq', 'pi', 'pieapp', 'piqe', 'psnr', 'psnry', 'qalign', 'qalign_4bit', 'qalign_8bit', 'qualiclip', 'qualiclip+', 'qualiclip+-clive', 'qualiclip+-flive', 'qualiclip+-spaq', 'sfid', 'ssim', 'ssimc', 'stlpips', 'stlpips-vgg', 'topiq_fr', 'topiq_fr-pipal', 'topiq_iaa', 'topiq_iaa_res50', 'topiq_nr', 'topiq

In [4]:
iqa_metric = pyiqa.create_metric('brisque', device=device)

In [29]:
import re

img_folder = '/root/exp/us-hand-to-large/results/infer_new/xijing_trainACropGray_vqdualv1_AtoB_flexNoResize'
img_paths = [os.path.join(img_folder, file) for file in os.listdir(img_folder)]

# 自然排序：按数字顺序排列
def natural_sort_key(path):
    # 提取文件名中的数字部分进行排序
    filename = os.path.basename(path)
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', filename)]

img_paths.sort(key=natural_sort_key)

score = iqa_metric(img_paths[0])
print(img_paths[0])
print(score)

/root/exp/us-hand-to-large/results/infer_new/xijing_trainACropGray_vqdualv1_AtoB_flexNoResize/xijing_Hand_video_3.png
tensor([51.1788], device='cuda:0')


In [30]:
# 批量计算IQA分数
from PIL import Image
import torchvision.transforms as transforms
import torch
import numpy as np
from tqdm import tqdm

# 图像预处理函数
def load_and_preprocess_images(img_paths, batch_size=8):
    """
    批量加载和预处理图像
    Args:
        img_paths: 图像路径列表
        batch_size: 批处理大小
    Returns:
        batches: 预处理后的图像批次列表
    """
    transform = transforms.Compose([
        transforms.ToTensor(),  # 转换为tensor并归一化到[0,1]
    ])
    
    batches = []
    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i:i+batch_size]
        batch_tensors = []
        
        for path in batch_paths:
            try:
                # 加载图像
                img = Image.open(path).convert('RGB')
                # 预处理
                img_tensor = transform(img)
                batch_tensors.append(img_tensor)
            except Exception as e:
                print(f"Error loading {path}: {e}")
                continue
        
        if batch_tensors:
            # 堆叠成批次 (N, 3, H, W)
            batch = torch.stack(batch_tensors, dim=0)
            batches.append(batch)
    
    return batches

# 批量计算IQA分数
def calculate_batch_iqa_scores(img_paths, iqa_metric, batch_size=8):
    """
    批量计算IQA分数
    Args:
        img_paths: 图像路径列表
        iqa_metric: IQA度量模型
        batch_size: 批处理大小
    Returns:
        scores: 分数列表
        valid_paths: 有效路径列表（与分数一一对应）
    """
    scores = []
    valid_paths = []
    
    # 预处理图像批次
    print("预处理图像...")
    batches = load_and_preprocess_images(img_paths, batch_size)
    
    # 批量计算分数
    print("计算IQA分数...")
    with torch.no_grad():
        for i, batch in enumerate(tqdm(batches, desc="Processing batches")):
            try:
                # 移动到设备
                batch = batch.to(device)
                # 计算分数
                batch_scores = iqa_metric(batch)
                
                # 记录分数和对应的路径
                start_idx = i * batch_size
                end_idx = min(start_idx + len(batch), len(img_paths))
                batch_paths = img_paths[start_idx:end_idx]
                
                # 如果返回的是tensor，转换为列表
                if isinstance(batch_scores, torch.Tensor):
                    if batch_scores.dim() == 0:  # 标量
                        batch_scores = [batch_scores.item()]
                    else:
                        batch_scores = batch_scores.cpu().numpy().tolist()
                
                scores.extend(batch_scores)
                valid_paths.extend(batch_paths)
                
            except Exception as e:
                print(f"Error processing batch {i}: {e}")
                continue
    
    return scores, valid_paths

# 执行批量计算
print(f"总共 {len(img_paths)} 张图像")
batch_size = 8  # 可以根据GPU内存调整
scores, valid_paths = calculate_batch_iqa_scores(img_paths, iqa_metric, batch_size)

print(f"\n成功计算了 {len(scores)} 张图像的IQA分数")
print(f"平均分数: {np.mean(scores):.4f}")
print(f"分数范围: {np.min(scores):.4f} - {np.max(scores):.4f}")

# 显示前几个结果
print("\n前5个结果:")
for i in range(min(5, len(scores))):
    filename = os.path.basename(valid_paths[i])
    print(f"{filename}: {scores[i]:.4f}")

总共 5376 张图像
预处理图像...
计算IQA分数...
计算IQA分数...


Processing batches: 100%|██████████| 672/672 [00:08<00:00, 77.69it/s]




成功计算了 5376 张图像的IQA分数
平均分数: 47.9889
分数范围: 13.4260 - 65.1198

前5个结果:
xijing_Hand_video_3.png: 51.1788
xijing_Hand_video_5.png: 43.7234
xijing_Hand_video_6.png: 43.5978
xijing_Hand_video_7.png: 45.3481
xijing_Hand_video_11.png: 49.0405
